# L4b: Breadth-First and Depth-First Search

__Which vertices can we reach, and in what order will we visit them?__ Breadth-first search (BFS) and depth-first search (DFS) answer these questions by following graph edges. They explore the same reachable vertices in different ways: breadth-first search proceeds in layers, while depth-first search follows one branch before returning to another.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * **Build an adjacency list:** Read a directed edge list and construct a sorted list of outgoing neighbors for each vertex. Explain why graph traversal uses the edge connections but ignores their weights.
> * **Implement DFS and BFS:** Complete recursive depth-first search and queue-based breadth-first search. Track visited vertices to prevent repeated exploration, and examine neighbors in ascending order to make the results reproducible.
> * **Compare traversal results:** Explain how the starting vertex and edge directions determine which vertices are reachable. Distinguish traversal order from a graph path, and test that both algorithms handle cycles and reject invalid starting vertices.

In this lab, we use the graph representations introduced in [L4a](../L4a/CHEME-5800-L4a-Lecture-GraphAndTreeRepresentations-Fall-2026.ipynb) to build an adjacency list, complete both traversal algorithms, and compare their results. We examine how edge directions and the starting vertex affect the search.

Let's get started!

___

## Algorithms

Both algorithms need to remember where to continue the search. Depth-first search uses the active recursive calls, while breadth-first search uses a queue. 

* __Depth-first search__ follows one branch recursively until it cannot continue, then returns to an earlier vertex to explore its remaining neighbors. [Read the DFS algorithm notebook](CHEME-5800-L4b-Algorithm-DepthFirstSearch-Fall-2026.ipynb).
* __Breadth-first search__ uses a first-in, first-out queue to explore vertices one edge from the start, then newly discovered vertices two edges away, and so on. [Read the BFS algorithm notebook](CHEME-5800-L4b-Algorithm-BreadthFirstSearch-Fall-2026.ipynb).

This difference in implementation can change the visit order even when the graph, starting vertex, and neighbor order are the same.
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines the data folder path, and loads the course package and student implementation.

Let's set up our code environment:


In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The setup loads the [`Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) for checks, [`DataFrames.jl`](https://dataframes.juliadata.org/stable/) for tables, and [`PrettyTables.jl`](https://ronisbr.github.io/PrettyTables.jl/stable/) for their display. It also loads the `L4bTraversal` module from [`src/Compute.jl`](src/Compute.jl).

After saving your changes to [`src/Compute.jl`](src/Compute.jl) in Task 2, rerun the setup cell to load the updated traversal functions. You do not need to restart the kernel.

___


## Task 1: Build and validate the directed graph
In this task, we will build the adjacency list used by both traversals. The file [`data/SimpleGraph.txt`](data/SimpleGraph.txt) describes the six-vertex directed graph shown below. Each record contains a source vertex, a target vertex, and an edge weight. We first load these records into the course graph model, then extract the outgoing neighbors of each vertex.

<div>
    <center>
        <img src="figs/Fig-Example-Graph.svg" width="680" alt="Six-vertex directed graph with seven weighted edges used to compare depth-first and breadth-first traversal"/>
    </center>
</div>

__What information does a traversal use?__ Both algorithms follow edge directions to find reachable vertices. Their adjacency list therefore records which vertices are connected, while the graph model retains the edge weights.

We supply [the `parse_edge_record(...)` function](docs/traversal-functions.md#parse_edge_record) to read the three fields from each data record. The file reader skips comments before passing records to this function:

In [ ]:
"""
    parse_edge_record(record::String, delimiter::Char = ',')

Parse `source,target,weight` text into the tuple expected by
`MyGraphEdgeModels(...)`. Return `nothing` if the field count is not three;
invalid numeric fields raise a parsing error.
"""
function parse_edge_record(record::String, delimiter::Char = ',')
    fields = strip.(split(record, delimiter)); # remove whitespace around the three fields
    length(fields) == 3 || return nothing

    source = parse(Int, fields[1]);       # directed-edge source identifier
    target = parse(Int, fields[2]);       # directed-edge target identifier
    weight = parse(Float64, fields[3]);   # edge cost; ignored by DFS and BFS
    return (source, target, weight)
end

We load the edge records with [the `MyGraphEdgeModels(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.MyGraphEdgeModels-Tuple%7BString%2C%20Function%7D) and construct the course graph model. We then pass the source-target pairs to [the `adjacency_from_edges(...)` function](docs/traversal-functions.md#adjacency_from_edges) to build the adjacency list.

Each vertex appearing in the edge file receives an entry, including vertices with no outgoing edges. The helper removes duplicate edges and sorts each vertex's outgoing neighbors.


In [ ]:
# Load the weighted edge records from the file shown in the schematic.
edge_file = joinpath(CHEME5800_L4B_DATA, "SimpleGraph.txt");
edge_models = MyGraphEdgeModels(edge_file, parse_edge_record; delim = ',', comment = '#');

# Retain the complete weighted representation in the course graph model.
directed_graph = build(MySimpleDirectedGraphModel, edge_models);

# Reduce each edge to connectivity only, then normalize the adjacency list.
edge_pairs = [(edge.source, edge.target) for edge in values(edge_models)];
adjacency = L4bTraversal.adjacency_from_edges(edge_pairs);

The table lists each vertex and its outgoing neighbors. An empty entry means the vertex has no outgoing edges.


In [ ]:
vertices = sort!(collect(keys(adjacency)));
adjacency_table = DataFrame(
    vertex = vertices,
    outgoing_neighbors = [isempty(adjacency[v]) ? "∅" : join(adjacency[v], ", ") for v in vertices],
);
pretty_table(adjacency_table)

__Does the representation match the graph?__ We check the six vertices, seven edges, and outgoing-neighbor list for each vertex.


In [ ]:
@testset "L4b directed-graph representation" begin
    # Check the two representations against the known six-vertex example.
    @test length(directed_graph.nodes) == 6
    @test length(directed_graph.edges) == 7
    @test vertices == collect(1:6)

    # Check every outgoing-neighbor vector used by DFS and BFS.
    @test adjacency == Dict(
        1 => [2, 3],
        2 => [3, 4],
        3 => [5],
        4 => [6],
        5 => [4],
        6 => Int64[],
    )
end;

Starting at vertex 1 reaches every vertex. Starting at vertex 3 reaches only $3\rightarrow5\rightarrow4\rightarrow6$; the edge directions prevent a return to vertices 1 or 2.

___


## Task 2: Implement DFS and BFS

In this task, we will complete recursive DFS and queue-based BFS, then compare their visit orders. Sorting outgoing neighbors makes each result reproducible.

The [`depth_first_order(...)`](docs/traversal-functions.md#depth_first_order) and [`breadth_first_order(...)`](docs/traversal-functions.md#breadth_first_order) functions are defined in [`src/Compute.jl`](src/Compute.jl). Both accept the same inputs and must meet the following requirements.

> __Traversal inputs and results:__
>
> __Inputs__
>
> * `adjacency::AbstractDict`: maps every vertex identifier to its outgoing neighbors. Neither function may mutate this dictionary or its neighbor collections.
> * `start::Integer`: the vertex at which traversal begins. A Boolean is not a valid identifier even though `Bool <: Integer` in Julia.
>
> __Output__
>
> * `Vector{Int64}`: every vertex reachable from `start`, recorded exactly once in traversal order. Examine outgoing neighbors in ascending identifier order so the result is reproducible.
>
> __Errors__
>
> * [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError): `start` is a Boolean or does not appear as a key in `adjacency`.

Use the [DFS algorithm notebook](CHEME-5800-L4b-Algorithm-DepthFirstSearch-Fall-2026.ipynb) and [BFS algorithm notebook](CHEME-5800-L4b-Algorithm-BreadthFirstSearch-Fall-2026.ipynb) while completing the six TODOs in [`src/Compute.jl`](src/Compute.jl):

1. Validate the DFS starting vertex and create an empty visited set and result vector.
2. Define the recursive helper: skip visited vertices; otherwise record the vertex and recursively explore its outgoing neighbors in ascending order.
3. Call the helper on the starting vertex and return the DFS order.
4. Validate the BFS starting vertex, initialize its visited set, result vector, and queue, then mark and enqueue the starting vertex.
5. Use a head index to process queued vertices and append each to the result vector.
6. Examine outgoing neighbors in ascending order; mark and enqueue each unvisited neighbor.

Save [`src/Compute.jl`](src/Compute.jl), then rerun the setup cell. Each unfinished traversal raises an error identifying the TODOs you need to complete.

In [ ]:
# Compute first-visit orders from the common starting vertex.
dfs_order = L4bTraversal.depth_first_order(adjacency, 1);
bfs_order = L4bTraversal.breadth_first_order(adjacency, 1);
(depth_first = dfs_order, breadth_first = bfs_order)

Both traversals visit all six vertices, but in different orders:

* **Depth-first:** Starting at vertex 1, we follow `1 → 2 → 3 → 5 → 4 → 6`. All six vertices are visited before backtracking begins, giving `[1, 2, 3, 5, 4, 6]`.
* **Breadth-first:** Processing vertex 1 adds vertices 2 and 3 to the queue. Processing those vertices adds 4 and 5, and processing vertex 4 adds 6. Removing vertices from the front of the queue gives `[1, 2, 3, 4, 5, 6]`.

___


## Task 3: Test and compare the traversals

In this task, we will test both implementations and compare the order in which they visit reachable vertices.

**Is a traversal order a path?** The breadth-first result lists vertex 4 immediately after vertex 3, but the graph has no edge $3\rightarrow4$. The sequence records the order of visits; it need not follow connected edges.

The table below compares the two traversals at each visit position.


In [ ]:
# Align the two traversal orders by visit position.
comparison_table = DataFrame(
    visit_position = collect(eachindex(dfs_order)),
    dfs_vertex = dfs_order,
    bfs_vertex = bfs_order,
    same_vertex = dfs_order .== bfs_order,
);
pretty_table(comparison_table)

Now start both traversals at vertex 3. Only the chain `3 → 5 → 4 → 6` is reachable, so both traversals should return `[3, 5, 4, 6]`. There are no competing branches to produce different visit orders.

In [ ]:
# Compare the reachable set and first-visit order from an interior vertex.
dfs_from_three = L4bTraversal.depth_first_order(adjacency, 3);
bfs_from_three = L4bTraversal.breadth_first_order(adjacency, 3);
(depth_first = dfs_from_three, breadth_first = bfs_from_three)

**What happens when the graph contains a cycle?** We construct the cycle $1\rightarrow2\rightarrow3\rightarrow1$ with an additional edge $2\rightarrow4$. When either traversal encounters vertex 1 again, the visited set prevents it from exploring that vertex again. Both traversals should finish with the order `[1, 2, 3, 4]`.


In [ ]:
# Build a cycle; the adjacency helper removes the repeated (1, 2) edge.
cyclic_adjacency = L4bTraversal.adjacency_from_edges([
    (1, 2), (1, 2), (2, 3), (3, 1), (2, 4),
]);
cyclic_snapshot = deepcopy(cyclic_adjacency); # preserve the input before either traversal runs

@testset "deterministic DFS and BFS contracts" begin
    # Verify the known first-visit orders for the lab graph.
    @test dfs_order == [1, 2, 3, 5, 4, 6]
    @test bfs_order == [1, 2, 3, 4, 5, 6]
    @test dfs_from_three == [3, 5, 4, 6]
    @test bfs_from_three == [3, 5, 4, 6]

    # Verify termination and first-visit behavior in a cyclic graph.
    @test L4bTraversal.depth_first_order(cyclic_adjacency, 1) == [1, 2, 3, 4]
    @test L4bTraversal.breadth_first_order(cyclic_adjacency, 1) == [1, 2, 3, 4]

    # Verify non-mutation and every documented invalid-start category.
    @test cyclic_adjacency == cyclic_snapshot
    @test_throws ArgumentError L4bTraversal.depth_first_order(adjacency, 99)
    @test_throws ArgumentError L4bTraversal.depth_first_order(adjacency, true)
    @test_throws ArgumentError L4bTraversal.breadth_first_order(adjacency, true)
    @test_throws ArgumentError L4bTraversal.breadth_first_order(adjacency, 99)
end;

___


## Summary

We built an adjacency list for a directed graph, implemented two traversals, and compared how they explore the vertices reachable from a chosen start.

> __Key Takeaways:__
>
> * **Recursion and queues:** We used recursive calls to follow one branch in depth-first search and a FIFO queue to explore layers in breadth-first search. The two methods returned different orders from vertex 1 while reaching the same six vertices.
>
> * **Tracking visited vertices:** We recorded discovered vertices to prevent repeated exploration when edges converge or form a cycle. For breadth-first search, marking each vertex as it enters the queue prevents duplicate queue entries.
>
> * **Interpreting traversal order:** We compared results from two starting vertices and showed why a visit sequence need not be a path. Breadth-first search visits vertices layer by layer, while depth-first search follows one branch before backtracking.

In [L4c](../L4c/CHEME-5800-L4c-Lecture-ShortestPathAlgorithms-Fall-2026.ipynb), we include edge weights and ask which route has the smallest total cost.
